In [66]:
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.1.20 langchain-community==0.0.38 langchain-core==0.1.52

Found existing installation: langchain 0.1.20
Uninstalling langchain-0.1.20:
  Successfully uninstalled langchain-0.1.20
Found existing installation: langchain-community 0.0.38
Uninstalling langchain-community-0.0.38:
  Successfully uninstalled langchain-community-0.0.38
Found existing installation: langchain-core 0.1.52
Uninstalling langchain-core-0.1.52:
  Successfully uninstalled langchain-core-0.1.52
  Using cached langchain-0.1.20-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Using cached langchain_core-0.1.52-py3-none-any.whl.metadata (5.9 kB)
Using cached langchain-0.1.20-py3-none-any.whl (1.0 MB)
Using cached langchain_community-0.0.38-py3-none-any.whl (2.0 MB)
Using cached langchain_core-0.1.52-py3-none-any.whl (302 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-checkpoin

In [67]:
!pip install langchain faiss-cpu sentence-transformers crewai groq deepeval

In [68]:
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

from crewai import Agent, Task, Crew
from crewai.tools import tool

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

from groq import Groq

In [69]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

Enter Groq API Key: ··········


In [70]:
text = """
Artificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks
that normally require human intelligence. These tasks include learning, reasoning, problem-solving, perception,
and language understanding. AI systems can be broadly categorized into narrow AI and general AI.

Machine Learning (ML) is a subset of AI that enables systems to learn from data without being explicitly programmed.
Deep Learning is a further subset of ML that uses neural networks with many layers.

Applications of AI include healthcare, autonomous vehicles, natural language processing, robotics, and finance.
AI can improve efficiency, reduce human error, and enable new capabilities. However, it also raises concerns about
bias, privacy, and job displacement.

Retrieval-Augmented Generation (RAG) is a technique that combines retrieval of documents with text generation.
It helps reduce hallucinations by grounding responses in real data.

Ethical considerations in AI include fairness, accountability, transparency, and safety.
"""

In [71]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = splitter.create_documents([text])

embedding = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [72]:
@tool
def retrieve_docs(query: str) -> str:
    """Retrieve relevant documents from the vector store based on query"""

    docs = vectorstore.similarity_search(query, k=3)
    return "\n".join([d.page_content for d in docs])

In [73]:
client = Groq()

def generate_answer(question, context):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Answer only using the context."},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content

In [74]:
rag_agent = Agent(
    role="RAG Agent",
    goal="Retrieve and answer questions",
    backstory="Expert in retrieval augmented generation",
    tools=[retrieve_docs],
    verbose=True
)

In [75]:
def rag_pipeline(question):
    context = retrieve_docs.run(question)
    answer = generate_answer(question, context)
    return {"question": question, "answer": answer, "context": context}

In [76]:
@tool
def evaluate_answer(question: str, answer: str, context: str):
    """Evaluate answer quality using simple logic (no OpenAI)"""

    # Check if answer uses context
    faithfulness_score = 0.8 if any(word in context.lower() for word in answer.lower().split()) else 0.5

    # Check if answer relates to question
    relevancy_score = 0.8 if any(word in answer.lower() for word in question.lower().split()) else 0.5

    verdict = "PASS" if (faithfulness_score > 0.7 and relevancy_score > 0.7) else "FAIL"

    return {
        "faithfulness": faithfulness_score,
        "relevancy": relevancy_score,
        "verdict": verdict
    }

In [77]:
eval_agent = Agent(
    role="Evaluator",
    goal="Evaluate answer quality",
    backstory="Expert in evaluation metrics",
    tools=[evaluate_answer],
    verbose=True
)

In [78]:
revisor_agent = Agent(
    role="Revisor",
    goal="Improve bad answers",
    backstory="Fixes incorrect answers",
    verbose=True
)

In [79]:
def revise_answer(question, answer, context):
    prompt = f"""
    Improve the answer using ONLY the context.

    Question: {question}
    Context: {context}
    Bad Answer: {answer}

    Give a better answer:
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [80]:
def full_pipeline(question):
    result = rag_pipeline(question)

    eval_result = evaluate_answer.run(
        question=result["question"],
        answer=result["answer"],
        context=result["context"]
    )

    if eval_result["verdict"] == "FAIL":
        revised = revise_answer(
            result["question"],
            result["answer"],
            result["context"]
        )

        eval_result2 = evaluate_answer.run(
            question=result["question"],
            answer=revised,
            context=result["context"]
        )

        return result, eval_result, revised, eval_result2

    return result, eval_result, None, None

In [81]:
questions = [
    "What is AI?",
    "What is machine learning?",
    "What is RAG?",
    "What are ethical concerns in AI?",
    "How does AI help healthcare?"
]

for q in questions:
    print("\nQUESTION:", q)
    res = full_pipeline(q)
    print(res)


QUESTION: What is AI?
({'question': 'What is AI?', 'answer': 'AI (Artificial Intelligence) is a field of computer science focused on creating systems that can perform tasks, and includes aspects such as machine learning and language understanding.', 'context': 'Artificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks\nMachine Learning (ML) is a subset of AI that enables systems to learn from data without being explicitly programmed.\nand language understanding. AI systems can be broadly categorized into narrow AI and general AI.'}, {'faithfulness': 0.8, 'relevancy': 0.8, 'verdict': 'PASS'}, None, None)

QUESTION: What is machine learning?
({'question': 'What is machine learning?', 'answer': 'Machine Learning (ML) is a subset of AI that enables systems to learn from data without being explicitly programmed.', 'context': 'Machine Learning (ML) is a subset of AI that enables systems to learn from data without being explicitly programm

In [82]:
bad_questions = [
    "Who is the president of Mars?",
    "What is quantum teleportation?"
]

for q in bad_questions:
    print("\nQUESTION:", q)
    res = full_pipeline(q)
    print(res)


QUESTION: Who is the president of Mars?
({'question': 'Who is the president of Mars?', 'answer': 'There is no mention of the president of Mars in the context. The context only discusses Artificial Intelligence (AI) and its related concepts.', 'context': 'that normally require human intelligence. These tasks include learning, reasoning, problem-solving, perception,\nArtificial Intelligence (AI) is a field of computer science focused on creating systems that can perform tasks\nEthical considerations in AI include fairness, accountability, transparency, and safety.'}, {'faithfulness': 0.8, 'relevancy': 0.8, 'verdict': 'PASS'}, None, None)

QUESTION: What is quantum teleportation?
({'question': 'What is quantum teleportation?', 'answer': 'There is no information about quantum teleportation in the context provided. The context only discusses Deep Learning, Artificial Intelligence, and Retrieval-Augmented Generation.', 'context': 'Deep Learning is a further subset of ML that uses neural net

## Reflection

In this assignment, we built a self-evaluating agentic RAG system that combines retrieval, evaluation, and revision using a multi-agent workflow. During testing, we observed that questions requiring precise factual grounding or multi-step reasoning caused the most failures. This was mainly because the retrieved context sometimes did not contain enough detailed information, leading the model to generate partially correct or vague answers.

The revision step proved to be effective in improving answer quality. When the evaluator flagged a response as failing, the revisor agent used the feedback and context to refine the answer, often increasing both faithfulness and relevancy scores. However, the improvement was not always perfect, especially when the original retrieved context itself was insufficient or ambiguous.

To improve reliability, one key change would be enhancing the retrieval stage by using better embeddings, increasing the number of retrieved documents, or applying reranking techniques. Additionally, integrating a stronger evaluation mechanism or multiple evaluation passes could further improve accuracy and robustness.

For future extensions, this system could be integrated with monitoring tools like TruLens to continuously track performance metrics such as hallucination rates and answer quality over time. This would allow for real-time feedback and continuous improvement of the system in production environments.